In [1]:
print("Here begins the hybrid models notebook")

Here begins the hybrid models notebook


In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
df_ml = pd.read_csv("../../data/processed/student_depression_ml.csv")

df_ml.head()

,Age,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Work/Study Hours,Financial Stress,CGPA__missing,Financial Stress__missing,Gender_female,Gender_male,City_3.0,City_agra,City_ahmedabad,City_bangalore,City_bhavna,City_bhopal,City_chennai,City_city,City_delhi,City_faridabad,City_gaurav,City_ghaziabad,City_harsh,City_harsha,City_hyderabad,City_indore,City_jaipur,City_kalyan,City_kanpur,City_khaziabad,City_kibara,City_kolkata,City_less delhi,City_less than 5 kalyan,City_lucknow,City_ludhiana,City_m.com,City_m.tech,City_me,City_meerut,City_mihir,City_mira,City_mumbai,City_nagpur,City_nalini,City_nalyan,City_nandini,City_nashik,City_patna,City_pune,City_rajkot,City_rashi,City_reyansh,City_saanvi,City_srinagar,City_surat,City_thane,City_vaanya,City_vadodara,City_varanasi,City_vasai-virar,City_visakhapatnam,Profession_architect,Profession_chef,Profession_civil engineer,Profession_content writer,Profession_digital marketer,Profession_doctor,Profession_educational consultant,Profession_entrepreneur,Profession_lawyer,Profession_manager,Profession_pharmacist,Profession_student,Profession_teacher,Profession_ux/ui designer,Sleep Duration_5-6 hours,Sleep Duration_7-8 hours,Sleep Duration_Less than 5 hours,Sleep Duration_More than 8 hours,Sleep Duration_Others,Dietary Habits_Healthy,Dietary Habits_Moderate,Dietary Habits_Others,Dietary Habits_Unhealthy,Degree_bachelor,Degree_doctorate,Degree_master,Degree_other,Degree_professional,Degree_school,Have you ever had suicidal thoughts ?_no,Have you ever had suicidal thoughts ?_yes,Family History of Mental Illness_no,Family History of Mental Illness_yes,Depression_yes
0,33.0,5.0,0.0,8.97,2.0,0.0,3.0,1.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,1
1,24.0,2.0,0.0,5.90,5.0,0.0,3.0,2.0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0
2,31.0,3.0,0.0,7.03,5.0,0.0,9.0,1.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0
3,28.0,3.0,0.0,5.59,2.0,0.0,4.0,5.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,1
4,25.0,4.0,0.0,8.13,3.0,0.0,1.0,1.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,0,0


In [2]:
# City is dropped due to severe data quality issues —
# the column contains personal names, degree abbreviations,
# and other non-geographic values indicating data entry errors
# at source. It cannot be reliably used as a predictor.
city_cols = [col for col in df_ml.columns if col.startswith("City_")]

df_ml = df_ml.drop(columns=city_cols)
df_ml.head()

,Age,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Work/Study Hours,Financial Stress,CGPA__missing,Financial Stress__missing,Gender_female,Gender_male,Profession_architect,Profession_chef,Profession_civil engineer,Profession_content writer,Profession_digital marketer,Profession_doctor,Profession_educational consultant,Profession_entrepreneur,Profession_lawyer,Profession_manager,Profession_pharmacist,Profession_student,Profession_teacher,Profession_ux/ui designer,Sleep Duration_5-6 hours,Sleep Duration_7-8 hours,Sleep Duration_Less than 5 hours,Sleep Duration_More than 8 hours,Sleep Duration_Others,Dietary Habits_Healthy,Dietary Habits_Moderate,Dietary Habits_Others,Dietary Habits_Unhealthy,Degree_bachelor,Degree_doctorate,Degree_master,Degree_other,Degree_professional,Degree_school,Have you ever had suicidal thoughts ?_no,Have you ever had suicidal thoughts ?_yes,Family History of Mental Illness_no,Family History of Mental Illness_yes,Depression_yes
0,33.0,5.0,0.0,8.97,2.0,0.0,3.0,1.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,1
1,24.0,2.0,0.0,5.90,5.0,0.0,3.0,2.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0
2,31.0,3.0,0.0,7.03,5.0,0.0,9.0,1.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0
3,28.0,3.0,0.0,5.59,2.0,0.0,4.0,5.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,1
4,25.0,4.0,0.0,8.13,3.0,0.0,1.0,1.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,0,0


In [3]:
# The dataset is titled "student depression" but contains mixed
# profession entries. Inspect distribution before filtering.
profession_cols = [col for col in df_ml.columns if col.startswith("Profession_")]

profession_counts = df_ml[profession_cols].sum().sort_values(ascending=False)
total = len(df_ml)

print("Profession distribution:")
for col, count in profession_counts.items():
    label = col.replace("Profession_", "")
    print(f"  {label:<30} {count:>5}  ({count/total:.1%})")

non_student = total - df_ml["Profession_student"].sum()
print(f"\nNon-student rows: {int(non_student)} ({non_student/total:.1%})")
print(f"Student rows:     {int(df_ml['Profession_student'].sum())} ({df_ml['Profession_student'].sum()/total:.1%})")


Profession distribution:
  student                        27870  (99.9%)
  architect                          8  (0.0%)
  teacher                            6  (0.0%)
  digital marketer                   3  (0.0%)
  chef                               2  (0.0%)
  content writer                     2  (0.0%)
  pharmacist                         2  (0.0%)
  doctor                             2  (0.0%)
  civil engineer                     1  (0.0%)
  educational consultant             1  (0.0%)
  manager                            1  (0.0%)
  lawyer                             1  (0.0%)
  entrepreneur                       1  (0.0%)
  ux/ui designer                     1  (0.0%)

Non-student rows: 31 (0.1%)
Student rows:     27870 (99.9%)


In [4]:
# Non-student respondents represent a different population context
# and are inconsistent with the dataset scope. Rows where
# Profession_student == 0 are dropped.
df_ml = df_ml[df_ml["Profession_student"] == 1].copy()

# Drop all Profession columns — no longer informative after filtering
profession_cols = [col for col in df_ml.columns if col.startswith("Profession_")]
df_ml = df_ml.drop(columns=profession_cols)

print(f"\nShape after filtering to students: {df_ml.shape}")

# Work Pressure is dropped — values (0, 2, 5) suggest data entry
# inconsistency, and work pressure is not applicable to a student
# population. Its associated missingness indicator is also dropped.
work_pressure_cols = [col for col in df_ml.columns
                      if "Work Pressure" in col or "work_pressure" in col.lower()]
df_ml = df_ml.drop(columns=work_pressure_cols, errors="ignore")

print(f"Dropped Work Pressure columns: {work_pressure_cols}")
print(f"Shape after dropping Work Pressure: {df_ml.shape}")


Shape after filtering to students: (27870, 32)
Dropped Work Pressure columns: ['Work Pressure']
Shape after dropping Work Pressure: (27870, 31)


In [5]:
# Inspect Job Satisfaction distribution
print("Job Satisfaction value counts:")
print(df_ml["Job Satisfaction"].value_counts().sort_index())
print(f"Zero values: {(df_ml['Job Satisfaction'] == 0).sum()} ({(df_ml['Job Satisfaction'] == 0).mean():.1%})")

# Job Satisfaction is dropped — not applicable to a student population.
# Value 0 likely encodes "not applicable" rather than true dissatisfaction,
# making the column uninterpretable in a student-only sample.
job_sat_cols = [col for col in df_ml.columns
                if "Job Satisfaction" in col or "job_satisfaction" in col.lower()]
df_ml = df_ml.drop(columns=job_sat_cols, errors="ignore")

print(f"Dropped Job Satisfaction columns: {job_sat_cols}")
print(f"Shape after dropping Job Satisfaction: {df_ml.shape}")

Job Satisfaction value counts:
Job Satisfaction
0.0    27862
1.0        2
2.0        3
3.0        1
4.0        2
Name: count, dtype: int64
Zero values: 27862 (100.0%)
Dropped Job Satisfaction columns: ['Job Satisfaction']
Shape after dropping Job Satisfaction: (27870, 30)


In [6]:
# One reference category dropped per OHE group to prevent
# perfect multicollinearity in linear models.
REFERENCE_COLS = [
    "Gender_male",                                # reference: male
    "Sleep Duration_7-8 hours",                   # reference: healthy sleep
    "Dietary Habits_Moderate",                    # reference: moderate diet
    "Degree_bachelor",                            # reference: bachelor degree
    "Have you ever had suicidal thoughts ?_no",   # reference: no suicidal thoughts
    "Family History of Mental Illness_no",        # reference: no family history
]

df_ml = df_ml.drop(columns=[c for c in REFERENCE_COLS if c in df_ml.columns])
print(f"\nShape after dropping reference categories: {df_ml.shape}")
df_ml.head()


Shape after dropping reference categories: (27870, 24)


,Age,Academic Pressure,CGPA,Study Satisfaction,Work/Study Hours,Financial Stress,CGPA__missing,Financial Stress__missing,Gender_female,Sleep Duration_5-6 hours,Sleep Duration_Less than 5 hours,Sleep Duration_More than 8 hours,Sleep Duration_Others,Dietary Habits_Healthy,Dietary Habits_Others,Dietary Habits_Unhealthy,Degree_doctorate,Degree_master,Degree_other,Degree_professional,Degree_school,Have you ever had suicidal thoughts ?_yes,Family History of Mental Illness_yes,Depression_yes
0,33.0,5.0,8.97,2.0,3.0,1.0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1
1,24.0,2.0,5.90,5.0,3.0,2.0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,31.0,3.0,7.03,5.0,9.0,1.0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0
3,28.0,3.0,5.59,2.0,4.0,5.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1
4,25.0,4.0,8.13,3.0,1.0,1.0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0


In [7]:
# Define feature matrix X and target vector y
# Depression_yes is the binary target variable -removed from X
X = df_ml.drop(columns=["Depression_yes"])
y = df_ml["Depression_yes"]

# Check dimensions of predictors and target
print(X.shape, y.shape)

# Check for missing values in the target should be 0
print("Missing values in target:", y.isna().sum())

# Display class distribution, check balance
print("\nTarget class distribution:")
print(y.value_counts())

print("\nTarget class distribution (normalized):")
print(y.value_counts(normalize=True))

(27870, 23) (27870,)
Missing values in target: 0

Target class distribution:
Depression_yes
1    16308
0    11562
Name: count, dtype: int64

Target class distribution (normalized):
Depression_yes
1    0.585145
0    0.414855
Name: proportion, dtype: float64


In [8]:
# Split the dataset into training and test sets
# test_size=0.2 means 80% training, 20% testing
# random_state ensures reproducibility
# stratify=y preserves class distribution in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display shapes of training and test sets
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Display class distribution in the training set
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

# Display class distribution in the test set
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Train shape: (22296, 23)
Test shape: (5574, 23)

Train class distribution:
Depression_yes
1    0.585127
0    0.414873
Name: proportion, dtype: float64

Test class distribution:
Depression_yes
1    0.585217
0    0.414783
Name: proportion, dtype: float64


In [9]:
# Initialize Random Forest model for feature selection
# class_weight="balanced": adjusts for class imbalance
# depth and split constraints help reduce overfitting
rf_selector = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# Fit Random Forest on the training data
rf_selector.fit(X_train, y_train)

# Select features with importance above the median importance
# This keeps approximately half of the most relevant predictors
selector = SelectFromModel(
    rf_selector,
    threshold="median"
)

# Fit selector and transform training/test data
selector.fit(X_train, y_train)
X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

# Retrieve names of selected features
selected_features = X_train.columns[selector.get_support()]

# Display selected features
print("Number of selected features:", len(selected_features))
print("\nSelected features:")
print(selected_features.tolist())

Number of selected features: 12

Selected features:
['Age', 'Academic Pressure', 'CGPA', 'Study Satisfaction', 'Work/Study Hours', 'Financial Stress', 'Sleep Duration_Less than 5 hours', 'Sleep Duration_More than 8 hours', 'Dietary Habits_Healthy', 'Dietary Habits_Unhealthy', 'Degree_school', 'Have you ever had suicidal thoughts ?_yes']


In [10]:
# Initialize Logistic Regression model
# class_weight="balanced": compensates for class imbalance
# max_iter increased to ensure convergence
logreg_hybrid = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# Train Logistic Regression on features selected by Random Forest
logreg_hybrid.fit(X_train_sel, y_train)

# Generate predictions on the test set
y_pred_hybrid1 = logreg_hybrid.predict(X_test_sel)

# Evaluate performance of Hybrid 1
print("Hybrid 1: RF Feature Selection -> Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_hybrid1))
print("F1:", f1_score(y_test, y_pred_hybrid1))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_hybrid1))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_hybrid1))

Hybrid 1: RF Feature Selection -> Logistic Regression
Accuracy: 0.8433799784714747
F1: 0.863912704598597

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.83      0.82      2312
           1       0.88      0.85      0.86      3262

    accuracy                           0.84      5574
   macro avg       0.84      0.84      0.84      5574
weighted avg       0.84      0.84      0.84      5574


Confusion Matrix:

[[1930  382]
 [ 491 2771]]


In [11]:
# Create a DataFrame of Logistic Regression coefficients
# For binary classification, there is one coefficient vector
coef_df = pd.DataFrame({
    "Feature": selected_features,
    "Coefficient": logreg_hybrid.coef_[0]
})

# Sort features by absolute coefficient value
# This highlights the strongest predictors of depression
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()
coef_df.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
11,Have you ever had suicidal thoughts ?_yes,2.482395,2.482395
1,Academic Pressure,0.835774,0.835774
9,Dietary Habits_Unhealthy,0.587712,0.587712
5,Financial Stress,0.562285,0.562285
8,Dietary Habits_Healthy,-0.501733,0.501733
6,Sleep Duration_Less than 5 hours,0.340935,0.340935
7,Sleep Duration_More than 8 hours,-0.281253,0.281253
3,Study Satisfaction,-0.255292,0.255292
10,Degree_school,-0.178899,0.178899
0,Age,-0.120130,0.120130


In [12]:
# Initialize XGBoost classifier for feature selection
# objective="binary:logistic": binary classification
# eval_metric="logloss": appropriate for binary classification
xgb_selector = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

# Fit XGBoost on training data
xgb_selector.fit(X_train, y_train)

# Select features with importance above the median
selector_xgb = SelectFromModel(
    xgb_selector,
    threshold="median"
)

# Fit selector and transform training/test data
selector_xgb.fit(X_train, y_train)
X_train_sel_xgb = selector_xgb.transform(X_train)
X_test_sel_xgb = selector_xgb.transform(X_test)

# Retrieve names of selected features
selected_features_xgb = X_train.columns[selector_xgb.get_support()]

# Display selected features
print("Number of selected features:", len(selected_features_xgb))
print("\nSelected features:")
print(selected_features_xgb.tolist())

Number of selected features: 12

Selected features:
['Age', 'Academic Pressure', 'Study Satisfaction', 'Work/Study Hours', 'Financial Stress', 'Sleep Duration_Less than 5 hours', 'Sleep Duration_More than 8 hours', 'Dietary Habits_Healthy', 'Dietary Habits_Unhealthy', 'Degree_school', 'Have you ever had suicidal thoughts ?_yes', 'Family History of Mental Illness_yes']


In [13]:
# Initialize Logistic Regression model for the second hybrid approach
logreg_hybrid_xgb = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# Train Logistic Regression on XGBoost-selected features
logreg_hybrid_xgb.fit(X_train_sel_xgb, y_train)

# Generate predictions on the test set
y_pred_hybrid2 = logreg_hybrid_xgb.predict(X_test_sel_xgb)

# Evaluate performance of Hybrid 2
print("Hybrid 2: XGB Feature Selection -> Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_hybrid2))
print("F1:", f1_score(y_test, y_pred_hybrid2))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_hybrid2))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_hybrid2))

Hybrid 2: XGB Feature Selection -> Logistic Regression
Accuracy: 0.8448152134912091
F1: 0.8650335465751288

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.84      0.82      2312
           1       0.88      0.85      0.87      3262

    accuracy                           0.84      5574
   macro avg       0.84      0.84      0.84      5574
weighted avg       0.85      0.84      0.85      5574


Confusion Matrix:

[[1937  375]
 [ 490 2772]]


In [16]:
coef_df_xgb = pd.DataFrame({
    "Feature": selected_features_xgb,
    "Coefficient": logreg_hybrid_xgb.coef_[0]
})

coef_df_xgb["Abs_Coefficient"] = coef_df_xgb["Coefficient"].abs()
coef_df_xgb.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
10,Have you ever had suicidal thoughts ?_yes,2.481113,2.481113
1,Academic Pressure,0.832948,0.832948
8,Dietary Habits_Unhealthy,0.585381,0.585381
4,Financial Stress,0.563622,0.563622
7,Dietary Habits_Healthy,-0.506204,0.506204
5,Sleep Duration_Less than 5 hours,0.337969,0.337969
6,Sleep Duration_More than 8 hours,-0.284390,0.284390
11,Family History of Mental Illness_yes,0.263245,0.263245
2,Study Satisfaction,-0.258373,0.258373
9,Degree_school,-0.175274,0.175274


In [14]:
# Define base models for stacking
# - Logistic Regression: interpretable linear model
# - Random Forest: nonlinear ensemble model
# - XGBoost: boosting model with strong predictive power
base_estimators = [
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )),
    ("rf", RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

# Initialize stacking classifier
# - final_estimator combines predictions from base models
# - class_weight="balanced"
# - stack_method="predict_proba" uses predicted probabilities as inputs
stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)

# Train stacking model
stack_model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_stack = stack_model.predict(X_test)

# Evaluate performance of Hybrid 3
print("Hybrid 3: Stacking")
print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print("F1:", f1_score(y_test, y_pred_stack))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stack))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_stack))

Hybrid 3: Stacking
Accuracy: 0.8464298528884104
F1: 0.8665835411471322

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.84      0.82      2312
           1       0.88      0.85      0.87      3262

    accuracy                           0.85      5574
   macro avg       0.84      0.85      0.84      5574
weighted avg       0.85      0.85      0.85      5574


Confusion Matrix:

[[1938  374]
 [ 482 2780]]


In [17]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
import pandas as pd

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "f1_macro":    make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted"),
}

hybrid_models = {
    "RF Selection → LogReg":    logreg_hybrid,
    "XGB Selection → LogReg":   logreg_hybrid_xgb,
    "Stacking (LR + RF + XGB)": stack_model,
}

cv_results_hybrid = {}
for name, model in hybrid_models.items():
    if name == "RF Selection → LogReg":
        X_cv_full = selector.transform(X)
    elif name == "XGB Selection → LogReg":
        X_cv_full = selector_xgb.transform(X)
    else:
        X_cv_full = X

    scores = cross_validate(model, X_cv_full, y, cv=cv, scoring=scoring, n_jobs=-1)

    y_prob_cv = cross_val_predict(
        model, X_cv_full, y,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    auc = roc_auc_score(y, y_prob_cv)

    cv_results_hybrid[name] = {
        "F1 Macro (mean)":    round(scores["test_f1_macro"].mean(), 3),
        "F1 Macro (std)":     round(scores["test_f1_macro"].std(),  3),
        "F1 Weighted (mean)": round(scores["test_f1_weighted"].mean(), 3),
        "F1 Weighted (std)":  round(scores["test_f1_weighted"].std(),  3),
        "AUC":                round(auc, 3),
    }

cv_hybrid_df = pd.DataFrame(cv_results_hybrid).T
print(cv_hybrid_df)

                          F1 Macro (mean)  F1 Macro (std)  F1 Weighted (mean)  \
RF Selection → LogReg               0.839           0.004               0.843   
XGB Selection → LogReg              0.840           0.005               0.845   
Stacking (LR + RF + XGB)            0.840           0.005               0.844   

                          F1 Weighted (std)    AUC  
RF Selection → LogReg                 0.004  0.921  
XGB Selection → LogReg                0.005  0.921  
Stacking (LR + RF + XGB)              0.005  0.921  


In [15]:
# Create a summary table comparing all hybrid approaches
results = pd.DataFrame([
    {
        "Model": "RF selection -> Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_hybrid1),
        "F1": f1_score(y_test, y_pred_hybrid1),
    },
    {
        "Model": "XGB selection -> Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_hybrid2),
        "F1": f1_score(y_test, y_pred_hybrid2),
    },
    {
        "Model": "Stacking (LR + RF + XGB)",
        "Accuracy": accuracy_score(y_test, y_pred_stack),
        "F1": f1_score(y_test, y_pred_stack),
    }
])

# Sort models by F1 score
results.sort_values("F1", ascending=False)

,Model,Accuracy,F1
2,Stacking (LR + RF + XGB),0.846430,0.866584
1,XGB selection -> Logistic Regression,0.844815,0.865034
0,RF selection -> Logistic Regression,0.843380,0.863913


In [18]:
results_hybrid = pd.DataFrame([
    {
        "Model":              "RF Selection → LogReg",
        "Macro F1 (test)":    round(f1_score(y_test, y_pred_hybrid1, average="macro"), 3),
        "Weighted F1 (test)": round(f1_score(y_test, y_pred_hybrid1, average="weighted"), 3),
        "CV Macro F1":        cv_hybrid_df.loc["RF Selection → LogReg", "F1 Macro (mean)"],
        "CV Macro F1 std":    cv_hybrid_df.loc["RF Selection → LogReg", "F1 Macro (std)"],
        "AUC":                cv_hybrid_df.loc["RF Selection → LogReg", "AUC"],
    },
    {
        "Model":              "XGB Selection → LogReg",
        "Macro F1 (test)":    round(f1_score(y_test, y_pred_hybrid2, average="macro"), 3),
        "Weighted F1 (test)": round(f1_score(y_test, y_pred_hybrid2, average="weighted"), 3),
        "CV Macro F1":        cv_hybrid_df.loc["XGB Selection → LogReg", "F1 Macro (mean)"],
        "CV Macro F1 std":    cv_hybrid_df.loc["XGB Selection → LogReg", "F1 Macro (std)"],
        "AUC":                cv_hybrid_df.loc["XGB Selection → LogReg", "AUC"],
    },
    {
        "Model":              "Stacking (LR + RF + XGB)",
        "Macro F1 (test)":    round(f1_score(y_test, y_pred_stack, average="macro"), 3),
        "Weighted F1 (test)": round(f1_score(y_test, y_pred_stack, average="weighted"), 3),
        "CV Macro F1":        cv_hybrid_df.loc["Stacking (LR + RF + XGB)", "F1 Macro (mean)"],
        "CV Macro F1 std":    cv_hybrid_df.loc["Stacking (LR + RF + XGB)", "F1 Macro (std)"],
        "AUC":                cv_hybrid_df.loc["Stacking (LR + RF + XGB)", "AUC"],
    },
]).set_index("Model")

print(results_hybrid)

                          Macro F1 (test)  Weighted F1 (test)  CV Macro F1  \
Model                                                                        
RF Selection → LogReg               0.840               0.844        0.839   
XGB Selection → LogReg              0.841               0.845        0.840   
Stacking (LR + RF + XGB)            0.843               0.847        0.840   

                          CV Macro F1 std    AUC  
Model                                             
RF Selection → LogReg               0.004  0.921  
XGB Selection → LogReg              0.005  0.921  
Stacking (LR + RF + XGB)            0.005  0.921  


In [ ]:
## Hybrid Models — Results Summary (Student Depression Dataset)

### Approach
# Three hybrid pipelines evaluated:
# 1. RF feature selection (threshold=median, 12 features) → Logistic Regression
# 2. XGB feature selection (threshold=median, 12 features) → Logistic Regression
# 3. Stacking ensemble (LR + RF + XGB base learners, LR meta-learner, cv=5)

### Key Findings
# - All three hybrid models achieve virtually identical performance
#   (CV Macro F1: 0.839–0.840, AUC: 0.921 for all three), indicating
#   that the choice of hybrid strategy is irrelevant for this dataset.
# - AUC of 0.921 is identical across all three models, suggesting the
#   learned decision boundary is the same regardless of approach.
# - Stacking marginally leads on test Macro F1 (0.843) but the
#   difference is negligible — not practically meaningful.
# - CV std is very low (0.004–0.005), confirming highly stable
#   generalization consistent with the large sample size.

### Comparison with Baseline Models
# Hybrid models do not outperform baseline models on this dataset.
# Best baseline (Logistic Regression/XGBoost, CV Macro F1: 0.841,
# AUC: 0.921) matches best hybrid (XGB→LogReg, 0.840, AUC: 0.921).
# This is a meaningful finding — feature selection and ensembling
# add no value when the signal is already strong and concentrated
# in a small number of features (suicidal ideation, academic
# pressure, financial stress).

### Feature Selection Stability
# Both RF and XGB selectors chose 12 features with high overlap:
# - Shared: Age, Academic Pressure, Study Satisfaction,
#   Work/Study Hours, Financial Stress, Sleep Duration extremes,
#   Dietary Habits extremes, Degree_school, suicidal thoughts
# - RF selected CGPA; XGB selected Family History instead
# This near-identical selection confirms the signal structure is
# stable and robust to the choice of selector.

### Comparison with MH Tech Hybrid Models
# Unlike MH Tech where hybrid models showed marginal improvement
# over baselines (CV Macro F1: 0.488 vs 0.470), here no improvement
# is observed. This is consistent with the ceiling effect — when
# baseline performance is already high (~0.841), there is little
# room for hybrid strategies to add value.